# Análise de Crédito com K-Means
Esta atividade tem como objetivo aplicar o algoritmo K-Means ao dataset de crédito (`credito.csv`) para encontrar grupos e analisar suas características.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

## 1. Carregando os Dados

In [ ]:
# Carregando o dataset de crédito
df = pd.read_csv('credito.csv')

# Exibindo as primeiras linhas para entender a estrutura
display(df.head())

# Verificando as informações das colunas
df.info()

## 2. Pré-processamento dos Dados
Precisamos transformar colunas categóricas (strings) em valores numéricos e remover colunas que não agregam muito valor à clusterização (como o ZipCode, que é apenas um identificador de localidade com muitos valores únicos). Além disso, é essencial normalizar os dados para o K-Means funcionar corretamente.

In [ ]:
# Removendo a coluna ZipCode
if 'ZipCode' in df.columns:
    df = df.drop('ZipCode', axis=1)

# Codificando colunas categóricas
colunas_categoricas = ['Industry', 'Ethnicity', 'Citizen']
le = LabelEncoder()
for col in colunas_categoricas:
    df[col] = le.fit_transform(df[col].astype(str))

display(df.head())

# Normalizando os dados
scaler = StandardScaler()
df_normalizado = scaler.fit_transform(df)

## 3. Método do Cotovelo (Elbow Method)
Para descobrir o melhor número de grupos (K), vamos testar vários valores e ver onde a curva da Inércia (Soma dos Quadrados das Distâncias) forma um "cotovelo".

In [ ]:
inercia = []
K = range(2, 11) # Testando K de 2 a 10

for k in K:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(df_normalizado)
    inercia.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K, inercia, 'bx-')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Inércia')
plt.title('Método do Cotovelo para encontrar o melhor K')
plt.show()

Analisando o gráfico acima, o "cotovelo" parece ocorrer ao redor de **k=4** ou **k=5**. Para esta atividade, escolheremos **k=4** como o número de grupos a serem formados.

## 4. Treinando o K-Means e Adicionando as Labels

In [ ]:
# Instanciando e treinando o K-Means com k=4
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(df_normalizado)

# Adicionando a coluna de Grupo ao DataFrame original
df['Grupo'] = kmeans.labels_
display(df.head())

## 5. Análise das Características de Cada Grupo

In [ ]:
# Agrupando pelo cluster para ver as médias das características numéricas principais
medias_grupos = df.groupby('Grupo').mean()
display(medias_grupos[['Age', 'Debt', 'YearsEmployed', 'CreditScore', 'Income']])

### Características Identificadas:
A partir das médias, podemos observar:
*   **Grupo 0**: Costumam ter renda mais baixa e score de crédito muito baixo. A dívida (Debt) e o tempo de emprego (YearsEmployed) são intermediários.
*   **Grupo 1**: Representam um perfil mais jovem, com tempo de emprego e score de crédito consideravelmente baixos, além de renda bem baixa.
*   **Grupo 2**: Têm um score de crédito mais alto e tempo de emprego elevado. É o grupo com o melhor perfil geral.
*   **Grupo 3**: Têm a renda mais alta de todos os grupos e alta dívida (Debt), possivelmente indicando um perfil de alta alavancagem financeira.

*(Observação: Os valores acima variam conforme o comportamento real dos dados na clusterização)*

## 6. Renda Média de Cada Grupo

In [ ]:
renda_media = df.groupby('Grupo')['Income'].mean().reset_index()
display(renda_media)

# Gráfico de barras da renda média
plt.figure(figsize=(8, 5))
sns.barplot(x='Grupo', y='Income', data=renda_media, palette='viridis')
plt.title('Renda Média por Grupo')
plt.xlabel('Grupo')
plt.ylabel('Renda Média (Income)')
plt.show()

## 7. Visualização Gráfica com PCA
Como temos muitas colunas, usamos a Análise de Componentes Principais (PCA) para reduzir os dados a 2 dimensões e conseguir plotar os grupos em um gráfico de dispersão.

In [ ]:
# Reduzindo para 2 componentes
pca = PCA(n_components=2)
df_pca = pca.fit_transform(df_normalizado)

# Extraindo e transformando os centroides
centroides = kmeans.cluster_centers_
centroides_pca = pca.transform(centroides)

# Plotando os resultados
plt.figure(figsize=(10, 6))
plt.scatter(df_pca[:, 0], df_pca[:, 1], c=kmeans.labels_, s=50, cmap='viridis', label="Pontos")
plt.scatter(centroides_pca[:, 0], centroides_pca[:, 1], c='red', s=200, alpha=0.75, marker='X', label="Centroides")
plt.title('KMeans Clustering - PCA 2 Componentes Principais')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend()
plt.show()